In [1]:
import os
import sys

if "google.colab" in sys.modules:
    # Running in Colab

    !git clone https://github.com/pthengtr/kcw-analytics.git
    !cd /content/kcw-analytics && git pull origin main

    from google.colab import drive
    drive.mount("/content/drive")

    BASE_FOLDER = "/content/drive/Shareddrives"
    BASE_FOLDER_GIT = "/content"
else:
    # Running in local Jupyter
    BASE_FOLDER = r"G:\Shared drives"
    BASE_FOLDER_GIT = r"C:\Users\Windows 11\Notebook"

print("Using folder:", BASE_FOLDER)

Cloning into 'kcw-analytics'...
remote: Enumerating objects: 873, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 873 (delta 1), reused 1 (delta 1), pack-reused 866 (from 1)
Receiving objects: 100% (873/873), 711.98 KiB | 2.75 MiB/s, done.
Resolving deltas: 100% (600/600), done.
From https://github.com/pthengtr/kcw-analytics
 * branch            main       -> FETCH_HEAD
Already up to date.
Mounted at /content/drive
Using folder: /content/drive/Shareddrives


In [2]:
import os
import pandas as pd

folder = "/content/drive/Shareddrives/KCW-Data/kcw_analytics/01_raw"

data = {}

for file in os.listdir(folder):
    if file.endswith(".csv"):
        path = os.path.join(folder, file)
        data[file] = pd.read_csv(
            path,
            dtype={
              "BCODE": "string",
              "ITEMNO": "string",
              "BILLNO": "string",
            },
            encoding="utf-8-sig",
            low_memory=False   # stops chunk guessing
        )
        print(f"Loaded: {file} -> {data[file].shape}")

Loaded: raw_inventory_hq_2024.csv -> (4983, 8)
Loaded: raw_syp_pimas_purchase_bills.csv -> (3047, 49)
Loaded: raw_syp_simas_sales_bills.csv -> (13506, 49)
Loaded: raw_syp_sidet_sales_lines.csv -> (39185, 38)
Loaded: raw_syp_pidet_purchase_lines.csv -> (28378, 41)
Loaded: raw_syp_icmas_products.csv -> (38057, 94)
Loaded: raw_hq_pidet_purchase_lines.csv -> (153013, 41)
Loaded: raw_hq_icmas_products.csv -> (115022, 94)
Loaded: raw_hq_pimas_purchase_bills.csv -> (49834, 49)
Loaded: raw_hq_sidet_sales_lines.csv -> (733921, 38)
Loaded: raw_hq_apmas_payable.csv -> (979, 20)
Loaded: raw_hq_armas_receivable.csv -> (2236, 20)
Loaded: raw_hq_simas_sales_bills.csv -> (276253, 49)
Loaded: raw_hq_pvmas_notes_vouchers.csv -> (13924, 32)
Loaded: raw_hq_rvmas_notes_vouchers.csv -> (13924, 32)


In [3]:
#"21050289", "22050086", "22050285"
df_hq_pidet = data["raw_hq_pidet_purchase_lines.csv"].copy()

#"21050289", "22010585", "22050259"
df_syp_pidet = data["raw_syp_pidet_purchase_lines.csv"].copy()


In [21]:
import numpy as np

df_hq_pidet["QTY"] = pd.to_numeric(df_hq_pidet["QTY"], errors="coerce")
df_hq_pidet["MTP"] = pd.to_numeric(df_hq_pidet["MTP"], errors="coerce")

df_hq_pidet["AMOUNT_NO_VAT"] = np.where(
    df_hq_pidet["TAXIC"].astype(str).str.upper() == "Y",
    df_hq_pidet["AMOUNT"] / 1.07,
    df_hq_pidet["AMOUNT"]
)
df_hq_pidet["PRICE"] = df_hq_pidet["AMOUNT_NO_VAT"] / (df_hq_pidet["QTY"] * df_hq_pidet["MTP"])

df_syp_pidet["QTY"] = pd.to_numeric(df_syp_pidet["QTY"], errors="coerce")
df_syp_pidet["MTP"] = pd.to_numeric(df_syp_pidet["MTP"], errors="coerce")

df_syp_pidet["AMOUNT_NO_VAT"] = np.where(
    df_syp_pidet["TAXIC"].astype(str).str.upper() == "Y",
    df_syp_pidet["AMOUNT"] / 1.07,
    df_syp_pidet["AMOUNT"]
)
df_syp_pidet["PRICE"] = df_syp_pidet["AMOUNT_NO_VAT"] / (df_syp_pidet["QTY"] * df_syp_pidet["MTP"])


In [22]:
df = df_hq_pidet

df["BILLDATE"] = pd.to_datetime(df["BILLDATE"], errors="coerce")

df_hq_21050289 = df[
    df["BCODE"].astype(str).isin(["21050289"]) &
    (df["BILLDATE"].dt.year == 2025)
]

df_hq_22050086 = df[
    df["BCODE"].astype(str).isin(["22050086"]) &
    (df["BILLDATE"].dt.year == 2025)
]

df_hq_22050285 = df[
    df["BCODE"].astype(str).isin(["22050285"]) &
    (df["BILLDATE"].dt.year == 2025)
]



In [23]:
df = df_syp_pidet

df["BILLDATE"] = pd.to_datetime(df["BILLDATE"], errors="coerce")

df_syp_21050289 = df[
    df["BCODE"].astype(str).isin(["21050289"]) &
    (df["BILLDATE"].dt.year == 2025)
]

df_syp_22050259 = df[
    df["BCODE"].astype(str).isin(["22050259"]) &
    (df["BILLDATE"].dt.year == 2025)
]

df_syp_22010585 = df[
    df["BCODE"].astype(str).isin(["22010585"]) &
    (df["BILLDATE"].dt.year == 2025)
]


In [24]:
def format_thai_date(series):
    s = pd.to_datetime(series, errors="coerce")
    return s.dt.strftime("%d/%m/") + (s.dt.year + 543).astype("Int64").astype(str)

In [25]:
def export_dfs_to_excel_same_mapping(dfs_dict, output_file, column_map, date_cols=None):

    date_cols = date_cols or []

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        for sheet_name, df in dfs_dict.items():

            cols = [c for c in column_map if c in df.columns]
            out = df[cols].copy()

            # format Thai date
            for c in date_cols:
                if c in out.columns:
                    s = pd.to_datetime(out[c], errors="coerce")
                    out[c] = s.dt.strftime("%d/%m/") + (s.dt.year + 543).astype("Int64").astype(str)

            out = out.rename(columns=column_map)

            out.to_excel(writer, sheet_name=sheet_name, index=False)

In [26]:
folder = os.path.join(
    BASE_FOLDER,
    "KCW-Data",
    "kcw_analytics",
    "04_outputs",
    "00_temp",
    f"report_2026_03_07.xlsx"
)

# ⭐ create directory if missing
os.makedirs(os.path.dirname(folder), exist_ok=True)

In [27]:
column_map = {
    "BCODE": "รหัสสินค้า",
    "BILLDATE": "วันที่",
    "BILLNO": "เลขที่บิล",
    "PRICE": "่ราคาต่อหน่วย",
    "QTY":"จำนวน",
    "UI":"หน่วย",
    "MTP":"บรรจุ",
    "AMOUNT_NO_VAT": "ยอดรวม",
}

dfs_dict = {
    "สำนักงานใหญ่ 22050285": df_hq_22050285,
    "สำนักงานใหญ่ 22050086": df_hq_22050086,
    "สำนักงานใหญ่ 21050289": df_hq_21050289,
    "สี่แยกพัฒนา 22010585": df_syp_22010585,
    "สี่แยกพัฒนา 22050259": df_syp_22050259,
    "สี่แยกพัฒนา 21050289": df_syp_21050289,
}

export_dfs_to_excel_same_mapping(
    dfs_dict=dfs_dict,
    output_file=folder,
    column_map=column_map,
    date_cols=["BILLDATE"]
)

In [28]:
df = df_syp_pidet

df["BILLDATE"] = pd.to_datetime(df["BILLDATE"], errors="coerce")

df_syp_tf = df[
    df["BILLNO"].astype(str).str.contains("TFV", case=False, na=False) &
    (pd.to_datetime(df["BILLDATE"]).dt.year == 2025)
]

In [29]:
df_syp_tf

,ID,JOURMODE,JOURTYPE,JOURDATE,BILLTYPE,BILLDATE,BILLNO,LINE,ITEMNO,BCODE,...,CHGAMT,ACCTNO,PAID,SALEDATE,SALENO,SALEPRICE,ACCT_NO,CANCELED,DONE,AMOUNT_NO_VAT
0,15,1,PJ,2025-03-10,1,2025-03-10,TFV6803-001,10,1,03050539,...,NaN,7KCW,N,NaN,NaN,NaN,7SCL,N,N,433.196262
1,16,1,PJ,2025-03-10,1,2025-03-10,TFV6803-001,20,2,08056372,...,NaN,7KCW,N,NaN,NaN,NaN,7SSY1,N,N,122.747664
11,28,1,PJ,2025-03-18,1,2025-03-18,TFV6803-002,120,12,03052695,...,NaN,7KCW,N,NaN,NaN,NaN,7GP,N,N,182.000000
12,29,1,PJ,2025-03-18,1,2025-03-18,TFV6803-002,130,13,23051947,...,NaN,7KCW,N,NaN,NaN,NaN,7GP,N,N,165.747664
13,17,1,PJ,2025-03-18,1,2025-03-18,TFV6803-002,10,1,03050145,...,NaN,7KCW,N,NaN,NaN,NaN,7SCL,N,N,199.794393
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24245,1448498,1,PJ,2025-12-31,1,2025-12-31,TFV6812-125,20,2,09050665,...,NaN,7KCW,N,NaN,NaN,NaN,7TYI,N,N,130.280374
24246,1448499,1,PJ,2025-12-31,1,2025-12-31,TFV6812-125,30,3,02050610,...,NaN,7KCW,N,NaN,NaN,NaN,7TYI,N,N,607.710280
24247,1448484,1,PJ,2025-12-31,1,2025-12-31,TFV6812-123,50,5,15010107,...,NaN,7KCW,N,NaN,NaN,NaN,7VP,N,N,878.504673
24248,1448485,1,PJ,2025-12-31,1,2025-12-31,TFV6812-123,60,6,15010788,...,NaN,7KCW,N,NaN,NaN,NaN,7VP,N,N,653.271028


In [30]:
df_syp_tf_grouped = (
    df_syp_tf
    .groupby("BILLNO", as_index=False)
    .agg(
        BILLDATE=("BILLDATE", "first"),   # take first date
        line_count=("BILLNO", "size"),    # count rows
        AMOUNT_NO_VAT=("AMOUNT_NO_VAT", "sum")          # sum amount
    )
)

In [31]:
df_syp_tf_grouped

,BILLNO,BILLDATE,line_count,AMOUNT_NO_VAT
0,CNTFV6806-001,2025-06-10,1,-747.401869
1,CNTFV6807-001,2025-07-02,1,-751.401869
2,CNTFV6808-001,2025-08-28,1,-2990.654206
3,CNTFV6809-001,2025-09-10,1,-1670.046729
4,CNTFV6809-002,2025-09-10,1,-3573.900000
...,...,...,...,...
1169,TFV6812-123,2025-12-31,14,7432.626168
1170,TFV6812-124,2025-12-31,3,3182.822430
1171,TFV6812-125,2025-12-31,12,5745.672897
1172,TFV6812-126,2025-12-31,2,309.878505


In [32]:
def export_dfs_to_excel(dfs_dict, output_file, column_map_by_sheet, date_cols=None):
    date_cols = date_cols or []

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        for sheet_name, df in dfs_dict.items():
            column_map = column_map_by_sheet[sheet_name]
            cols = [c for c in column_map if c in df.columns]
            out = df[cols].copy()

            for c in date_cols:
                if c in out.columns:
                    s = pd.to_datetime(out[c], errors="coerce")
                    out[c] = s.dt.strftime("%d/%m/") + (s.dt.year + 543).astype("Int64").astype(str)

            out = out.rename(columns=column_map)
            out.to_excel(writer, sheet_name=sheet_name, index=False)

In [33]:
dfs_dict = {
    "TFV Bills": df_syp_tf_grouped,
    "TFV Lines": df_syp_tf,
}

In [34]:
column_map_by_sheet = {

    "TFV Bills": {
        "BILLNO": "เลขที่บิล",
        "BILLDATE": "วันที่",
        "line_count": "จำนวนรายการ",
        "AMOUNT_NO_VAT": "ยอดรวม",
    },

    "TFV Lines": {
        "BILLDATE": "วันที่",
        "BILLNO": "เลขที่บิล",
        "BCODE": "รหัสสินค้า",
        "PRICE": "่ราคาต่อหน่วย",
        "QTY":"จำนวน",
        "UI":"หน่วย",
        "MTP":"บรรจุ",
        "AMOUNT_NO_VAT": "ยอดรวม",
    },
}

In [35]:
folder = os.path.join(
    BASE_FOLDER,
    "KCW-Data",
    "kcw_analytics",
    "04_outputs",
    "00_temp",
    f"tfv_2026_03_07.xlsx"
)

# ⭐ create directory if missing
os.makedirs(os.path.dirname(folder), exist_ok=True)

In [36]:
export_dfs_to_excel(
    dfs_dict,
    folder,
    column_map_by_sheet,
    date_cols=["BILLDATE"]
)